# EDA из `main.ipynb`

Здесь оставлена только EDA-часть: загрузка таблицы, распределение классов, топ-5 классов и быстрый просмотр первых записей.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

CSV_PATH = Path("../data/extracted_audio/metadata.csv")
LABEL_MAPPING_PATH = Path("../data/extracted_audio/label_mapping.csv")
LABEL_COL = "label"
CATEGORY_COL = "category_id"

df = pd.read_csv(CSV_PATH)

if CATEGORY_COL not in df.columns:
    if LABEL_MAPPING_PATH.exists() and LABEL_COL in df.columns:
        mapping_df = pd.read_csv(LABEL_MAPPING_PATH)
        label_to_id = dict(zip(mapping_df["label"], mapping_df["class_id"]))
        df[CATEGORY_COL] = df[LABEL_COL].astype(str).map(label_to_id)
    elif LABEL_COL in df.columns:
        classes = sorted(df[LABEL_COL].astype(str).unique().tolist())
        label_to_id = {label: idx for idx, label in enumerate(classes)}
        df[CATEGORY_COL] = df[LABEL_COL].astype(str).map(label_to_id)
    else:
        raise ValueError(f"В таблице нет ни '{CATEGORY_COL}', ни '{LABEL_COL}'")

df = df.dropna(subset=[CATEGORY_COL]).copy()
df[CATEGORY_COL] = df[CATEGORY_COL].astype(int)

display(df.head())
display(
    df.groupby([CATEGORY_COL, LABEL_COL], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["count", CATEGORY_COL], ascending=[False, True])
    .head(10)
)
print("Всего строк:", len(df))
print("Уникальных классов:", df[CATEGORY_COL].nunique())


In [ ]:
def plot_category_distribution(
    dataframe: pd.DataFrame,
    column: str,
    title: str = None,
    color: str = "#00BCD4",
    bg_color: str = "#000066",
    top_n: int = None,
    save_path: str = None,
) -> go.Figure:
    if column not in dataframe.columns:
        raise ValueError(f"Колонка '{column}' не найдена в DataFrame")

    fig = go.Figure()

    if top_n is None:
        fig.add_trace(
            go.Histogram(
                x=dataframe[column],
                name=column,
                marker_color=color,
                opacity=0.75,
                hovertemplate="<b>Категория:</b> %{x}<br><b>Частота:</b> %{y}<extra></extra>",
            )
        )
        min_val = int(dataframe[column].min())
        max_val = int(dataframe[column].max())
        fig.update_xaxes(
            tickmode="linear",
            dtick=1,
            range=[min_val - 0.5, max_val + 0.5],
            tickangle=45,
            tickfont=dict(size=10),
        )
        bargap = 0.1
    else:
        top_data = dataframe[column].value_counts().head(top_n)
        fig.add_trace(
            go.Bar(
                x=top_data.index.astype(str),
                y=top_data.values,
                name=f"Top {top_n}",
                marker_color=color,
                opacity=0.85,
                hovertemplate="<b>Категория:</b> %{x}<br><b>Частота:</b> %{y}<extra></extra>",
            )
        )
        fig.update_xaxes(tickangle=0, tickfont=dict(size=12))
        bargap = 0.3

    fig.update_layout(
        title={
            "text": title or (f"Топ-{top_n} категорий" if top_n else "Распределение категорий"),
            "y": 0.95,
            "x": 0.5,
            "xanchor": "center",
            "yanchor": "top",
            "font": dict(size=24, family="Arial Black", color="#00BCD4"),
        },
        xaxis_title="ID Категории",
        yaxis_title="Количество записей",
        paper_bgcolor=bg_color,
        plot_bgcolor=bg_color,
        xaxis=dict(showgrid=False, gridcolor="#1DAAE6", linecolor="#00BCD4"),
        yaxis=dict(showgrid=True, gridcolor="#1DAAE6", linecolor="#00BCD4"),
        hovermode="x",
        bargap=bargap,
        height=600,
        width=1000,
        font=dict(family="Roboto, sans-serif", size=14, color="#00BCD4"),
    )

    if save_path:
        fig.write_html(save_path)
        print(f"График сохранен в {save_path}")

    return fig


In [ ]:
fig_all = plot_category_distribution(
    df,
    CATEGORY_COL,
    title="Распределение категорий в датасете",
)
fig_all.show()


In [ ]:
fig_top5 = plot_category_distribution(
    df,
    CATEGORY_COL,
    top_n=5,
    title="Топ-5 самых частых категорий",
)
fig_top5.show()


In [ ]:
subset_size = min(100, len(df))
subset_data = df.iloc[:subset_size].copy()
fig_first_rows = go.Figure()
fig_first_rows.add_trace(
    go.Bar(
        x=list(range(len(subset_data))),
        y=subset_data[CATEGORY_COL],
        marker_color="#00BCD4",
        marker_line_color="#000066",
        marker_line_width=1.5,
        opacity=0.85,
        hovertemplate="<b>Строка:</b> %{x}<br><b>Значение:</b> %{y}<extra></extra>",
    )
)
fig_first_rows.update_layout(
    title={
        "text": f"График по первым {subset_size} строкам",
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": dict(size=24, family="Arial Black", color="#00BCD4"),
    },
    xaxis_title="Номер строки",
    yaxis_title="ID Категории",
    paper_bgcolor="#000066",
    plot_bgcolor="#000066",
    xaxis=dict(showgrid=False, gridcolor="#1DAAE6", linecolor="#00BCD4", tickfont=dict(color="#00BCD4")),
    yaxis=dict(showgrid=True, gridcolor="#1DAAE6", linecolor="#00BCD4", tickfont=dict(color="#00BCD4")),
    bargap=0.5,
    bargroupgap=0.001,
    hovermode="x",
    height=500,
    width=1000,
    font=dict(family="Roboto, sans-serif", size=14, color="#00BCD4"),
)
fig_first_rows.show()
